In [0]:
from pyspark.sql import functions as F

CATALOG = "voebem"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

TABELA_SILVER_VRA = f"{CATALOG}.{SCHEMA_SILVER}.vra"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}"
)

df_vra = spark.table(TABELA_SILVER_VRA)

print(f"Registros disponíveis: {df_vra.count():,}")
print(f"Colunas disponíveis: {len(df_vra.columns)}")

### Agregação por empresa aérea

Consolidamos os dados limpos da camada **Silver** para calcular o volume de operações e a média de atrasos agrupados por companhia aérea (`icao_empresa`).

##### Métricas Calculadas:
* **`total_voos`**: Quantidade total de voos registrados por empresa (`count(*)`).
* **`media_atraso_partida_min`**: Média de atraso na partida em minutos (`avg(atraso_partida_min)`).
* **`media_atraso_chegada_min`**: Média de atraso na chegada em minutos (`avg(atraso_chegada_min)`).

In [0]:
df_gold_companhias = (
    df_vra
    .groupBy("icao_empresa")
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.round(
            F.avg("atraso_chegada_min"), 2
        ).alias("atraso_medio_chegada_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

%md
### Taxa de atrasos e ordenação

Calculamos o percentual de voos que sofreram atrasos superiores a 15 minutos em relação ao total de partidas válidas registradas por empresa:

* **`percentual_atrasados_15min`**: Proporção percentual de voos com atraso significativo (`(voos_atrasados_15min / voos_com_dados_partida) * 100`), arredondada para 2 casas decimais.
* **Tratamento de Divisão por Zero**: A métrica só é calculada quando `voos_com_dados_partida > 0`; caso contrário, é retornado `null`.
* **Ordenação**: O resultado é ordenado de forma decrescente pelo volume geral de voos (`total_voos`).

In [0]:
df_gold_companhias = (
    df_gold_companhias
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None)) # Cria um valor literal NULL
    )
    .orderBy(F.desc("total_voos"))
)

In [0]:
TABELA_GOLD_COMPANHIAS = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_companhias"
)

(
    df_gold_companhias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_COMPANHIAS)
)
display(df_gold_companhias)

%md
### Agregação por rota (origem x destino)

Agrupamos os dados por par de aeroportos (`icao_origem` e `icao_destino`) para consolidar o desempenho operacional das rotas de voo:

* **`total_voos`**: Quantidade total de registros por rota (`count(*)`).
* **`voos_com_dados_partida` / `voos_com_dados_chegada`**: Volume de voos que possuem informações válidas de atraso na partida e na chegada.
* **`atraso_medio_partida_min` / `atraso_medio_chegada_min`**: Média dos atrasos em minutos para partida e chegada, arredondada em 2 casas decimais.
* **`voos_atrasados_15min`**: Total de voos da rota que sofreram atraso superior a 15 minutos na partida.

In [0]:
df_gold_rotas = (
    df_vra
    .groupBy(
        "icao_origem",
        "icao_destino"
    )
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.count("atraso_chegada_min").alias(
            "voos_com_dados_chegada"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.round(
            F.avg("atraso_chegada_min"), 2
        ).alias("atraso_medio_chegada_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

%md
### Taxa de atraso por rota e ordenação

Calculamos o percentual de voos atrasados por rota e ordenamos o resultado pelas rotas mais movimentadas:

* **`percentual_atrasados_15min`**: Proporção percentual de voos da rota com atraso na partida superior a 15 minutos (`(voos_atrasados_15min / voos_com_dados_partida) * 100`), arredondada para 2 casas decimais.
* **Tratamento de Divisão por Zero**: O cálculo é executado apenas quando `voos_com_dados_partida > 0`; caso contrário, é atribuído `null`.
* **Ordenação**: As rotas são ordenadas de forma decrescente pelo volume total de voos (`total_voos`).

In [0]:
df_gold_rotas = (
    df_gold_rotas
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(F.desc("total_voos"))
)


In [0]:
TABELA_GOLD_ROTAS = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_rotas"
)

(
    df_gold_rotas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_ROTAS)
)
display(df_gold_rotas)

%md
### Agregação por aeroporto de origem

Consolidamos as métricas de desempenho de partidas agrupadas pelo aeroporto de origem (`icao_origem`):

* **`total_voos`**: Quantidade total de voos que decolaram do aeroporto (`count(*)`).
* **`voos_com_dados_partida`**: Total de registros que possuem horário/atraso de partida preenchido.
* **`atraso_medio_partida_min`**: Média do tempo de atraso na partida em minutos, arredondada para 2 casas decimais.
* **`voos_atrasados_15min`**: Quantidade de partidas que sofreram atraso superior a 15 minutos.

In [0]:
df_gold_aeroportos = (
    df_vra
    .groupBy("icao_origem")
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

### Taxa de atraso por aeroporto e ordenação

Calculamos o percentual de voos atrasados por aeroporto de origem e ordenamos os aeroportos por volume total de operações:

* **`percentual_atrasados_15min`**: Proporção percentual de voos do aeroporto com atraso na partida superior a 15 minutos (`(voos_atrasados_15min / voos_com_dados_partida) * 100`), arredondada para 2 casas decimais.
* **Tratamento de Divisão por Zero**: O cálculo é executado apenas quando `voos_com_dados_partida > 0`; caso contrário, é atribuído `null`.
* **Ordenação**: Os aeroportos são ordenados de forma decrescente pelo volume total de voos (`total_voos`).

In [0]:
df_gold_aeroportos = (
    df_gold_aeroportos
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(F.desc("total_voos"))
)

In [0]:
TABELA_GOLD_AEROPORTOS = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_aeroportos_origem"
)

(
    df_gold_aeroportos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_AEROPORTOS)
)
display(df_gold_aeroportos)

### Agregação mensal das operações

Consolidamos as métricas de desempenho de voos agrupadas por ano e mês de partida (`partida_prevista_ts`):

* **Filtro de Validação**: Consideramos apenas os registros que possuem a data e hora de partida prevista preenchidas (`isNotNull()`).
* **Enriquecimento Temporal**: Extraímos as colunas `ano`, `mes` e criamos a chave de agrupamento `ano_mes` no formato `yyyy-MM`.
* **`total_voos`**: Quantidade total de voos agendados no mês (`count(*)`).
* **`voos_com_dados_partida / voos_com_dados_chegada`**: Volume de registros com informações válidas de atraso na partida e na chegada.
* **`atraso_medio_partida_min / atraso_medio_chegada_min`**: Média dos atrasos em minutos na partida e na chegada, arredondada para 2 casas decimais.
* **`voos_atrasados_15min`**: Total de voos no mês que sofreram atraso superior a 15 minutos na partida.

In [0]:
df_gold_mensal = (
    df_vra
    .filter(F.col("partida_prevista_ts").isNotNull())
    .withColumn(
        "ano",
        F.year("partida_prevista_ts")
    )
    .withColumn(
        "mes",
        F.month("partida_prevista_ts")
    )
    .withColumn(
        "ano_mes",
        F.date_format(
            "partida_prevista_ts",
            "yyyy-MM"
        )
    )
    .groupBy(
        "ano",
        "mes",
        "ano_mes"
    )
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.count("atraso_chegada_min").alias(
            "voos_com_dados_chegada"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.round(
            F.avg("atraso_chegada_min"), 2
        ).alias("atraso_medio_chegada_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

### Taxa de atraso mensal e ordenação cronológica

Calculamos o percentual de voos atrasados no mês e ordenamos os resultados por ano e mês:

* **`percentual_atrasados_15min`**: Proporção percentual de voos do mês com atraso na partida superior a 15 minutos (`(voos_atrasados_15min / voos_com_dados_partida) * 100`), arredondada para 2 casas decimais.
* **Tratamento de Divisão por Zero**: O cálculo é executado apenas quando `voos_com_dados_partida > 0`; caso contrário, é atribuído `null`.
* **Ordenação**: Os registros são ordenados de forma crescente por `ano` e `mes`.

In [0]:
df_gold_mensal = (
    df_gold_mensal
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(
        "ano",
        "mes"
    )
)

In [0]:
TABELA_GOLD_MENSAL = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_mensal"
)

(
    df_gold_mensal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_MENSAL)
)
display(df_gold_mensal)

### Validação e auditoria das tabelas Gold

Iteramos sobre a lista de tabelas criadas no esquema `gold` para verificar e exibir a volumetria de registros e o total de colunas de cada dataset consolidado:

* **Controle de Qualidade**: Garante a integridade das tabelas `kpi_companhias`, `kpi_rotas`, `kpi_aeroportos_origem` e `kpi_mensal`.
* **Métricas Exibidas**: Retorna o nome da tabela, a contagem total de registros (`count()`) e a quantidade de colunas (`len(columns)`).

In [0]:
tabelas_gold = [
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_companhias",
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_rotas",
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_aeroportos_origem",
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_mensal"
]
for tabela in tabelas_gold:
    df = spark.table(tabela)

    print(f"\nTabela: {tabela}")
    print(f"Registros: {df.count():,}")
    print(f"Colunas: {len(df.columns)}")
